In [17]:
visualization_target = input("INPUT 'visualization_target RUN_ID'(e.g., run_id_6): ")
model_in_run = input("INPUT 'model_in_run'(e.g., bert_based, lda, tag): ")

In [18]:
from gc import collect

import pickle
import lib.stats.stats as st
from lib.utils.file_io import *
from lib.utils.statistics import *
from lib.utils.settings import set_matplotlib
from lib.visualization.distribution_collector import (collect_topic_distributions,
                                get_top_and_bottom_topics,
                                extract_specific_topics,
                                collect_tag_distributions)
# from constants import CONSTANTS
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.nonparametric.smoothers_lowess import lowess
import matplotlib as mpl

import psycopg2
import pandas as pd
import numpy as np
from lib.utils.statistics import *
# import config.config as conf
import datetime
import re
# 포뮬러 구성
import statsmodels.formula.api as smf
import numpy as np
import pandas as pd
import matplotlib as mpl
import lib.visualization.font_setting as font_setting
mpl.rcParams['font.family'] = font_setting.init_font()

from setting_for_sda.path_setting import path_list


Helvetica33-ExtendedThin /home/mghan/.fonts/Helvetica/Helvetica Extended Thin.ttf
Registered font name: Helvetica33-ExtendedThin


In [19]:
path_list

{'data_root_dir': '/mnt/hdd/mghan/so_data_availability',
 'bert_monthly_data_dir': '/mnt/hdd/mghan/so_data_availability/result/bert_based/run_id_0/data',
 'bert_monthly_data_dir_2': '/mnt/hdd/mghan/so_data_availability/result/bert_based/run_id_2/data',
 'bert_monthly_data_dir_3': '/mnt/hdd/mghan/so_data_availability/result/bert_based/run_id_3/data',
 'tag_monthly_data_dir': '/mnt/hdd/mghan/so_data_availability/result/tag/run_id_0/data',
 'tag_monthly_data_dir_2': '/mnt/hdd/mghan/so_data_availability/result/tag/run_id_2/data',
 'tag_monthly_data_dir_2_py': '/mnt/hdd/mghan/so_data_availability/result/tag/run_id_2/python/data',
 'tag_monthly_data_dir_2_cpp': '/mnt/hdd/mghan/so_data_availability/result/tag/run_id_2/cpp/data',
 'lda_monthly_data_dir': '/mnt/hdd/mghan/so_data_availability/result/lda/run_id_1/data',
 'bert_difficulty_data_dir': '/mnt/hdd/mghan/so_data_availability/result/bert_based/difficulty_annotated/data',
 'lda_difficulty_data_dir': '/mnt/hdd/mghan/so_data_availability/re

In [ ]:
viz_dir = f'{path_list["data_root_dir"]}/result/{model_in_run}/{visualization_target}'
data_dir = f"{viz_dir}/data"
option_dict = load_json(f"{viz_dir}/option.json")


output_dir = './fig/'
date_range = 'Weekly'



FileNotFoundError: [Errno 2] No such file or directory: '/mnt/hdd/mghan/so_data_availability/result/tag/run_id_6/option.json'

In [ ]:
df = pd.DataFrame(rows, columns = [
  'lang'
, 'cdate' 
, 'id' 
, 'tag'
, 'cnt'
, 'tot_cnt'
, 'pct'
])

In [ ]:
std_date = datetime.datetime(2022, 11, 30)
pre_std_date = datetime.datetime(2021, 12, 1)

In [ ]:
df['cdate'] = pd.to_datetime(df['cdate'], format="%Y-%m-%d")

In [ ]:
# a = df_did[(df_did['post_rel_week'] >=-20) & (df_did['post_rel_week'] <20)  ]
# b = df_did[(df_did['pre_rel_week'] >=-20) & (df_did['pre_rel_week'] <20)  ]


In [ ]:
df['rel_week'] = np.floor((df['cdate']-std_date).dt.days/7)

In [ ]:
df_py = df[df['lang']=='python']
df_cpp = df[df['lang']=='c++']
df_java = df[df['lang']=='java']
df_vba = df[df['lang']=='vba']

In [ ]:
# tag_dict = {
#             'python' : list(df_py['tag'].unique()), 
#             'c++' : list(df_cpp['tag'].unique()), 

#             }

# json.dumps(tag_dict)
# with open('../visualization/data/tag_by_lang.json', 'w') as f : 
# 	json.dump(tag_dict, f, indent=4)
	

In [ ]:
def pp_for_tag_df(df):
    # 상위, 하위 태그 리스트 추출
    df_bf_pro = df[df['rel_week']<0].groupby(['tag']).sum(['pct'])['pct'].sort_values().reset_index()
    tagnum = int(np.floor(df_bf_pro.shape[0]*0.2))
    bot_tag = list(df_bf_pro.iloc[:tagnum, 0])
    top_tag = list(df_bf_pro.iloc[tagnum:, 0])

    # stackedbar를 위한 계산 수행 (전체 대비 비율 계산 및 주차별 합산 계산)
    df_tot = df.groupby(['rel_week']).sum(['pct'])['pct'].reset_index(name = 'tot_pct')
    df_pct = pd.merge(df, df_tot, on = 'rel_week')

    df_pct['pct_pct'] = df_pct['pct']/df_pct['tot_pct']

    df_pct_bot = df_pct[df_pct['tag'].isin(bot_tag)]
    df_pct_top = df_pct[df_pct['tag'].isin(top_tag)]

    df_pct_top_tot = df_pct_top.groupby(['rel_week']).sum(['pct_pct'])['pct_pct'].reset_index()
    df_pct_bot_tot = df_pct_bot.groupby(['rel_week']).sum(['pct_pct'])['pct_pct'].reset_index()

    return {'Top 20% Tags' : df_pct_top_tot,   'Bottom 20% Tags' : df_pct_bot_tot}




In [ ]:
proportion_dict = dict()

In [ ]:
proportion_dict['python'] = pp_for_tag_df(df_py)
proportion_dict['cpp'] = pp_for_tag_df(df_cpp)
proportion_dict['java'] = pp_for_tag_df(df_java)
proportion_dict['vba'] = pp_for_tag_df(df_vba)

In [ ]:
color_list = ["#4C704C", "#A3C9A8"]

In [ ]:
proportion_dict.keys()

In [ ]:
sharey = False ## 또는 sharey=False
sharex = True ## 또는 sharex=False
g_num  = len(proportion_dict['python'].items())

fig, axs = plt.subplots(1, 8, figsize = (48, 6), constrained_layout=True)
alpha_list = [0.6, 0.5]
color_list = ["#a6d96a", "#1a9850"]

lang_key = list(proportion_dict.keys())

for x in range(len(lang_key)) : 
    for idx, (title, proportion) in enumerate(proportion_dict[lang_key[x]].items()):
        n_len = len(proportion_dict[lang_key[x]].items())
        rel_week = list(proportion['rel_week'])
        values = list(proportion['pct_pct'])

        i_idx = x*n_len + idx
        c_idx = i_idx% 2

        axs[i_idx].bar(rel_week, values, color=color_list[c_idx], width=1.0, align='center', alpha=alpha_list[c_idx]
        )
        axs[i_idx].axvline(x=0, color='tab:red', linestyle='-.', linewidth=1)

        if idx ==2 :
            axs[i_idx].set_ylim(0.85, 1.0)
            axs[i_idx].set_yticks(np.arange(0.85, 1.01, 0.05))

        axs[i_idx].set_title(f'{title} for {lang_key[x]}', fontsize=25)
        axs[i_idx].tick_params(axis='x', labelsize=16)
        axs[i_idx].tick_params(axis='y', labelsize=16)



axs[0].set_ylabel("Accumulated tag share", fontsize = 22)

fig.supxlabel("Week relative to ChatGPT release", fontsize=22) 
plt.savefig(f"{output_dir}C_Result_Fig2_1.png", dpi=300, bbox_inches='tight')
plt.show();

In [ ]:
tag_proportion_dict = dict()
for key in CONSTANTS.lang_tag_dict.keys():
    tag_dir = f'../../visualization/{CONSTANTS.tag_monthly_data_dir_2[3:-5]}/data/{key}'
    file_list = os.listdir(tag_dir)
    df = pd.DataFrame()
    for f_nm in file_list:
        tmp = pd.DataFrame(load_json(f'{tag_dir}/{f_nm}'))
        df = pd.concat([df, tmp], axis = 0)
    tag_proportion_dict[key] = df

In [ ]:
tag_proportion_dict['python']

In [ ]:
std_date = datetime.datetime(2022, 11, 30)
tag_distribution_dict = dict()

for key in CONSTANTS.lang_tag_dict.keys():
    df = tag_proportion_dict[key]
    df['cdate'] = pd.to_datetime(df['creationdate'], format="%Y-%m-%d")
    df['rel_day'] = (df['cdate'] - std_date).dt.days
    df['rel_week'] = np.floor(df['rel_day']/7)
    df = df[(df['rel_week']>-53 ) & (df['rel_week']<=52 )]
    df.head()

    tot_df = df.groupby(['rel_week', 'matched_tag'])['cnt'].sum().reset_index(name='tot_cnt')
    tag_dis_by_df = df.groupby(['rel_week', 'matched_tag', 'tag'])['cnt'].sum().reset_index(name = 'cnt')

    df_proportion = pd.merge(tag_dis_by_df, tot_df, on = ['rel_week', 'matched_tag'], how = 'left')
    df_proportion['proportion'] = df_proportion['cnt'] / df_proportion['tot_cnt']
    tag_distribution_dict[key] = df_proportion




In [ ]:
tag_distribution_dict['python']

In [ ]:
gini_dict = dict()
entropy_dict = dict()
for key in CONSTANTS.lang_tag_dict.keys():
    gini_dict[key] = list(map(lambda x: calculate_gini(list(x.values())), tag_distribution_dict[key][]))
    entropy_dict[key] = list(map(lambda x: calculate_entropy(list(x.values())), tag_distribution_dict[key]))

In [ ]:
print(np.mean(entropy_dict['cpp'][:52]))
print(np.mean(entropy_dict['cpp'][52:]))

In [ ]:
rel_week = np.array(np.arange(-52, 104))

In [ ]:
sharey = False ## 또는 sharey=False
sharex = True ## 또는 sharex=False

fig, axs = plt.subplots(4, 3, figsize = (24, 24), constrained_layout=True)
alpha_list = [0.6, 0.5]
color_list = ["#a6d96a", "#1a9850"]

for row, key in enumerate(CONSTANTS.lang_tag_dict.keys()):
    for x, (title, proportion) in enumerate(proportion_dict[key].items()):
        rel_week = list(proportion['rel_week'])
        values = list(proportion['pct_pct'])
        
        axs[row][x].bar(rel_week, values, color=color_list[x], width=1.0, align='center', alpha=alpha_list[x]
        )
        axs[row][x].axvline(x=0, color='tab:red', linestyle='-.', linewidth=1)

        if x ==0 :
            axs[row][x].set_ylim(0.85, 1.0)
            axs[row][x].set_yticks(np.arange(0.85, 1.01, 0.05))


        axs[row][x].text(0.5, 1.05, f"{title} for {key}",
                ha='center', va='bottom', fontsize=22, fontweight='bold', transform=axs[row][x].transAxes)

        axs[row][x].text(0.5, 1.00, "",
            ha='center', va='bottom', fontsize=15, transform=axs[row][x].transAxes)  
        axs[row][x].tick_params(axis='x', labelsize=16)
        axs[row][x].tick_params(axis='y', labelsize=16)


    idx = 2
    list_ = entropy_dict[key]
    x_rel, divider = get_dist_x_param_div(list_, 52)

    reg_bf = calc_regression_with_ci(x_rel[:divider], list_[:divider])
    reg_af = calc_regression_with_ci(x_rel[divider:], list_[divider:])

    reg_bf_summary = reg_bf["pred_summary"]
    reg_af_summary = reg_af["pred_summary"]

    # 회귀선 (예측값)
    reg_bf_y_pred = reg_bf_summary["mean"]
    reg_af_y_pred = reg_af_summary["mean"]
    # 신뢰구간
    reg_bf_ci_lower = reg_bf_summary["mean_ci_lower"]
    reg_bf_ci_upper = reg_bf_summary["mean_ci_upper"]

    reg_af_ci_lower = reg_af_summary["mean_ci_lower"]
    reg_af_ci_upper = reg_af_summary["mean_ci_upper"]

    # p_value_txt = '($p < 0.001$)' if p_value_0 <0.001 else '($p = {p_value_0:.3f}$)'

    axs[row][idx].scatter(x_rel, list_, color = 'darkgray', alpha = 0.7,  s=10, marker='x')
    axs[row][idx].axvline(x=0, color='tab:red', linestyle='-.', linewidth=1)
    # axs[idx].set_ylabel(f"{measure} of Topic Distribution", fontsize = 10)
    axs[row][idx].plot(x_rel[:divider], reg_bf_y_pred, linewidth=2, label = 'before ChatGPT')
    axs[row][idx].plot(x_rel[divider:], reg_af_y_pred, linewidth=2, label = 'after ChatGPT')

    axs[row][idx].fill_between(x_rel[:divider], reg_bf_ci_lower, reg_bf_ci_upper, alpha=0.1)
    axs[row][idx].fill_between(x_rel[divider:], reg_af_ci_lower, reg_af_ci_upper, alpha=0.1)

    axs[row][idx].legend(frameon=False, loc='best', fontsize=14)
    axs[row][idx].text(0.5, 1.05, f"Changes in Entropy (tag) for {key}",
                ha='center', va='bottom', fontsize=22, fontweight='bold', transform=axs[row][idx].transAxes)



    axs[row][idx].tick_params(axis='x', labelsize=16)
    axs[row][idx].tick_params(axis='y', labelsize=16)


    axs[row][0].set_ylabel("Accumulated tag share", fontsize = 22)
    axs[row][2].set_ylabel(f"Entropy", fontsize = 22)

fig.supxlabel("Weeks relative to ChatGPT release", fontsize=22) 
plt.savefig(f"{output_dir}C_Result_Fig2_2.png", dpi=300, bbox_inches='tight')
plt.show();

In [ ]:
tag_distribution_dict['cpp']